In [1]:
import tomllib
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
config

{'base_url': 'http://localhost:1234/v1',
 'model': 'openai/gpt-oss-20b',
 'big_model': 'openai/gpt-oss-120b',
 'unthinking': 'meta/llama-3.3-70b',
 'embedding_model': 'text-embedding-embeddinggemma-300m-qat',
 'api_key': 'local'}

# 05 · MCP: It's Just JSON-RPC

MCP is a **standardised JSON-RPC 2.0 protocol** that lets LLM clients discover and call tools on external servers.

```
Client (LLM host)          MCP Server (your tool)
   ──────────────────────────────────────────
   initialize          →
                       ←   capabilities
   tools/list          →
                       ←   [{name, description, inputSchema}, ...]
   tools/call          →   {name: "X", arguments: {...}}
                       ←   {content: [{type:"text", text:"result"}]}
```

That's it. No magic. The transport is usually stdio (subprocess pipes) or HTTP SSE.

Run `mcp_server.py`

Run `mcp_proxy.py`

In [12]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

In [14]:
client = streamablehttp_client('http://localhost:8080/mcp')
async with client as (read_stream, write_stream, *_):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()
        resp = await session.list_tools()
        for tool in resp.tools:
            print(tool.name)
            print(tool.description)
            print(tool.inputSchema)
            print('-' * 80)

        await session.call_tool('a_tool', arguments={'arg1': 'argument'})

a_tool

{'properties': {'arg1': {'title': 'Arg1', 'type': 'string'}}, 'required': ['arg1'], 'title': 'a_toolArguments', 'type': 'object'}
--------------------------------------------------------------------------------


## What MCP adds over raw tool calls

| Raw tool calls | MCP |
|---|---|
| Tools defined in API request | Tools discovered at runtime from server |
| Caller must know tool schema | Client auto-discovers via `tools/list` |
| No standard transport | Stdio or HTTP SSE with versioned handshake |
| No resource/prompt primitives | Adds `resources/list`, `prompts/list` |
| One-off per integration | Standard interface: one client, many servers |

MCP is essentially **a plugin interface standard** for LLM tools. The protocol is thin — the value is ecosystem standardisation.